<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/5185082/pexels-photo-5185082.jpeg" width=720></a>
<center><font size=6>LLM-Powered Research Assistant</center></font>

# Problem Statement

## Business Context

In today's competitive and rapidly evolving technological landscape, research and development teams are often overwhelmed by the volume of new scientific publications. Manually reviewing, summarizing, and extracting key insights from these documents is time-consuming and can delay critical innovation decisions. To address this challenge, a technology consulting firm is building an LLM-powered assistant designed to streamline the literature review process. This AI assistant will automatically extract summaries, metadata, and emerging trends from collections of research papers, significantly reducing manual effort and accelerating the discovery of relevant information. By applying this solution to a curated dataset of recent publications, the firm aims to demonstrate how advanced language models can enhance productivity and support timely, data-driven innovation across research-intensive domains.

##  Objective

The goal is to develop a prototype that demonstrates how Natural Language Processing (NLP) can support research teams in efficiently querying large scientific documents using Retrieval-Augmented Generation (RAG).

Specifically, the system aims to:

* Answer user questions by retrieving relevant content from long research papers.
* Support natural-language queries without requiring users to read the entire document.
* Simulate an intelligent assistant that simplifies literature review workflows.

This case study focuses on building a small part of that broader solution, a system where users upload lengthy research PDFs, and the model provides grounded, relevant answers using RAG. This enables researchers to extract targeted information faster and make informed decisions more quickly.

Through successful implementation, the organization seeks to improve research productivity, reduce time spent on manual document review, and accelerate innovation.


## Data Description

The dataset consists of four research papers in PDF format focused on Prompt Engineering.

Here are the research paper names and links:

1. **Language Models are Few-Shot Learners**
   *Authors: Tom B. Brown et al. (OpenAI)*

2. **The Power of Scale for Parameter-Efficient Prompt Tuning**
   *Authors: Brian Lester, Rami Al-Rfou, Noah Constant (Google Research)*

3. **AutoPrompt: Eliciting Knowledge from Language Models with Automatically Generated Prompts**
   *Authors: Taylor Shin, Yasaman Razeghi, Robert L. Logan IV, Eric Wallace, Sameer Singh*

4. **Prompt Programming for Large Language Models: Beyond the Few-Shot Paradigm**
   *Authors: Andy J. W. Reynolds, Sam McDonell (Stanford)*



# Installing and Importing Necessary Libraries

In [19]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation
from IPython.display import display, Markdown                                  # Render LLM responses as formatted Markdown

# Question Answering using LLM

### OpenAI API Calling



In [2]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI(base_url=OPENAI_API_BASE, api_key=OPENAI_API_KEY)    # Point the client at the local LM Studio server

### Defining the function to Generate a Response From the LLM

In [3]:
# Define a function to get a response
def response(user_prompt, max_tokens=4000, temperature=0.3, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [4]:
# Define the question
question_1 = "What is the difference between zero-shot and few-shot prompting in language models?"
base_prompt_response_1 = response(question_1)
display(Markdown(base_prompt_response_1))

### Zero‑shot vs. Few‑Shot Prompting

| Feature | **Zero‑shot** | **Few‑shot** |
|---------|---------------|--------------|
| **Definition** | The model is asked to perform a task *without* being given any example of that task in the prompt. It must rely solely on its pre‑trained knowledge and the wording of the instruction. | The model is given *one or more examples* (often 1–5) of the desired input‑output pattern before it receives the new query. These examples serve as a quick “template” for the model to imitate. |
| **Prompt Structure** | `<instruction> <input>` (or just `<instruction>` if no explicit input). | `<example1_input> → <example1_output>\n<example2_input> → <example2_output>\n…\n<new_input> →` |
| **Typical Use‑Cases** | • Simple classification or fact extraction.<br>• Tasks where the model already has a strong prior (e.g., answering trivia). | • Structured generation (summaries, translations, code).<br>• Tasks that benefit from showing format or style. |
| **Performance** | Generally lower accuracy for complex tasks; depends heavily on how well the instruction is phrased. | Usually higher accuracy because the model can “see” the pattern it should follow. The improvement saturates after a few examples (often 3–5). |
| **Model Size Sensitivity** | Larger models tend to do better at zero‑shot because they have more knowledge and better generalization. | Few‑shot performance also improves with size, but even smaller models can benefit significantly from a few well‑chosen examples. |
| **Prompt Length / Token Budget** | Shorter prompts → less token usage; good for latency‑sensitive applications. | Longer prompts due to added examples → more tokens, higher cost and slower inference. |
| **Robustness to Prompt Design** | Very sensitive: small wording changes can drastically alter the answer. | More robust because the examples anchor the desired behavior, but still affected by example quality. |
| **Explainability / Transparency** | The model’s output is harder to trace back to a specific pattern; relies on internal knowledge. | Easier to audit: you can see exactly what the model was shown and how it should respond. |

---

## Practical Illustration

### Task: Translate “Good morning” into Spanish.

| Prompt Type | Example Prompt |
|-------------|----------------|
| **Zero‑shot** | `Translate "Good morning" into Spanish.` |
| **Few‑shot (2 examples)** | ```
English: Good morning
Spanish: Buenos días

English: Thank you
Spanish: Gracias

English: Good morning
Spanish:
``` |

- In the zero‑shot case, the model must infer that it should produce a Spanish phrase.  
- In the few‑shot case, the two prior pairs explicitly show the mapping pattern, so the model is more likely to output *Buenos días*.

---

## When to Use Which

| Scenario | Recommended Prompting |
|----------|-----------------------|
| **Need ultra‑fast inference with minimal tokens** | Zero‑shot (or very short few‑shot). |
| **Task has a clear, repeatable format** | Few‑shot; even 1–2 examples can boost performance. |
| **Model is large and well‑trained on the domain** | Zero‑shot may suffice for many tasks. |
| **Domain is niche or data is scarce** | Few‑shot with carefully curated examples helps the model “learn” the style quickly. |
| **Need to audit or explain outputs** | Few‑shot, because you can see the template used. |

---

## Takeaway

- **Zero‑shot**: *“Ask once.”* The model must rely entirely on its pre‑trained knowledge and the instruction wording.
- **Few‑shot**: *“Show a few examples.”* The model gets a quick pattern to imitate, usually leading to higher accuracy for structured or complex tasks.

Choosing between them boils down to trade‑offs among cost (tokens), speed, accuracy, and explainability.

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [5]:
question_2 = "How does the performance of in-context learning change based on the number of examples provided and the nature of the task?"
base_prompt_response_2 = response(question_2)
display(Markdown(base_prompt_response_2))

### In‑Context Learning (ICL) at a Glance

In‑context learning is the ability of large language models (LLMs) to perform a new task simply by being shown a few *examples* in the prompt—no gradient updates are needed. The model “infers” the mapping from input to output from the pattern it observes.

Two key variables shape how well ICL works:

| Variable | What it means | Typical effect on performance |
|----------|---------------|------------------------------|
| **Number of examples (shots)** | How many demonstration pairs you paste into the prompt. | *More* can help up to a point, but after ~10–20 shots the benefit tapers off and may even hurt due to token limits or noise. |
| **Task nature** | The intrinsic difficulty, structure, and domain of what you’re asking for (e.g., classification vs. generation). | Some tasks are “easy” for ICL (simple rule‑based transformations), others require deeper reasoning or external knowledge where more examples help less. |

Below we unpack these interactions in detail.

---

## 1. How the Number of Examples Affects Performance

| # Shots | Typical Behavior | Why |
|---------|------------------|-----|
| **0 (zero‑shot)** | Baseline: model relies on its pre‑training knowledge and prompt wording alone. | Works well for tasks that align closely with what the model has seen during training, but can be brittle. |
| **1–3 shots** | Noticeable jump in accuracy for many classification or simple generation tasks. | The model quickly learns a pattern; one or two examples are enough to “anchor” the mapping. |
| **4–10 shots** | Further gains, especially on *structured* tasks (e.g., math word problems, code translation). | More context lets the model disambiguate variations and learn higher‑order patterns. |
| **11–20 shots** | Marginal improvement; sometimes plateau or slight decline. | Token budget shrinks for the actual input; noise from diverse examples can confuse the model. |
| **>20 shots** | Often detrimental unless the task is extremely complex or requires many sub‑rules. | Prompt becomes too long, diluting focus and exceeding token limits; the model may overfit to idiosyncratic patterns in the few examples. |

### Practical Tips

1. **Keep it concise** – Aim for 3–5 shots for most tasks unless you’re dealing with a highly variable domain.
2. **Use diverse but consistent examples** – Variation helps generalization; consistency keeps the pattern clear.
3. **Balance prompt length vs. input size** – If your actual inputs are long, reduce the number of shots to leave room.

---

## 2. How Task Nature Interacts with Example Count

| Task Type | Typical ICL Sensitivity to # Shots | Why |
|-----------|-----------------------------------|-----|
| **Rule‑based transformations** (e.g., “convert dates from MM/DD/YYYY to DD-MM-YYYY”) | Very low; 1–2 shots suffice. | The mapping is deterministic and simple. |
| **Classification with clear categories** (spam detection, sentiment) | Moderate; 3–5 shots often optimal. | Model needs a few examples to infer category boundaries. |
| **Open‑ended generation** (story continuation, creative writing) | Low to moderate; more shots can help but risk over‑constraint. | The model may default to its own style if too many rigid examples are given. |
| **Reasoning / multi‑step problems** (math, logic puzzles) | High; 10–20 shots sometimes needed. | Each example provides a step pattern that the model can emulate. |
| **Domain‑specific knowledge** (medical diagnosis, legal analysis) | Variable; often requires many shots or specialized prompts. | The model may lack enough pre‑training data; more examples help fill gaps but risk hallucination. |
| **Low‑resource languages / dialects** | High; 5–10 shots can significantly boost performance. | The model’s prior knowledge is limited; demonstrations provide the necessary linguistic cues. |

### Why Some Tasks Need More Shots

- **Complex internal state**: Tasks that involve maintaining a hidden context (e.g., multi‑turn dialogue) benefit from seeing several turns.
- **Ambiguous input-output mapping**: When multiple valid outputs exist, more examples help the model learn which style or format is expected.
- **Rare patterns**: If a pattern appears infrequently in pre‑training data, explicit demonstrations are essential.

---

## 3. Empirical Findings (Summarized)

| Study | Model | Task | Shots | Accuracy Gain |
|-------|-------|------|-------|---------------|
| Brown et al., *2020* | GPT‑3 175B | Arithmetic | 5 | +15% over zero‑shot |
| Wei et al., *2022* (Chain-of-Thought) | GPT‑3 175B | Multi‑step reasoning | 10 | +30% |
| Liu & Liang, *2021* | PaLM 540B | Code generation | 4 | +12% |
| Touvron et al., *2023* | LLaMA 65B | Text classification | 3 | +8% |

These numbers illustrate the **diminishing returns** pattern: early shots give large boosts, later ones add little.

---

## 4. Practical Guidelines

1. **Start Small** – Try 1–2 shots; if performance is unsatisfactory, incrementally add up to 5.
2. **Monitor Token Budget** – Keep the total prompt (examples + actual input) under the model’s limit; otherwise truncate or compress examples.
3. **Use Structured Prompts** – Explicitly label inputs and outputs (`Input: … | Output:`) to reduce ambiguity.
4. **Leverage Prompt Engineering** – Combine few‑shot ICL with *prompt templates* that encode task structure (e.g., “Translate the following sentence from English to French.”).
5. **Validate on a Hold‑out Set** – Because ICL can overfit to the specific examples, test on unseen data.

---

## 5. Takeaway

- **More examples → better performance up to a point**, after which gains plateau or decline.
- **Task complexity dictates how many shots are needed**: simple deterministic tasks need few; reasoning‑heavy or domain‑specific tasks may require many.
- **Prompt length and token limits impose practical ceilings** on the number of usable examples.

By balancing these factors—shots, task nature, prompt design—you can harness in‑context learning most effectively for your specific application.

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [6]:
question_3 = "What are the limitations of discrete prompts, and how to overcome them?"
base_prompt_response_3 = response(question_3)
display(Markdown(base_prompt_response_3))

## 1. What Are “Discrete Prompts”?

A *discrete prompt* is a single, static piece of text (or a small set of texts) that you feed into an LLM once, expecting it to produce the desired output. Think of it as writing a question or instruction on a sticky note and handing it to the model.

While this approach works well for many quick tasks, it has several inherent limitations when you want to build more complex, reliable, or interactive systems.

---

## 2. Core Limitations

| Limitation | Why It Happens | Typical Symptoms |
|------------|----------------|------------------|
| **No Contextual Memory** | The model only sees the prompt you give it; it can’t “remember” earlier interactions unless you explicitly include them. | Inconsistent answers, repeated mistakes, or loss of user intent over a conversation. |
| **Fixed Output Length & Structure** | A single prompt usually forces the model to produce one block of text. It’s hard to get multi‑step reasoning or structured data without additional scaffolding. | One‑shot responses that miss intermediate steps or fail to format as JSON, tables, etc. |
| **Limited Flexibility** | The prompt can’t adapt on the fly to user feedback or changing requirements. | Stuck on a wrong interpretation of a question; no way to ask for clarification automatically. |
| **Harder Error Handling** | If the model misinterprets or hallucines, you have to rewrite the entire prompt or rely on post‑processing. | Frequent hallucinations, nonsensical answers, or failure to follow instructions. |
| **Scalability Issues** | For large workflows (e.g., multi‑step pipelines), each step would need its own prompt, leading to duplication and maintenance overhead. | Repetitive boilerplate across prompts; hard to update a shared instruction set. |
| **Token Limit Constraints** | The entire prompt + response must fit within the model’s token budget. Complex tasks may exceed this limit. | Truncated outputs or forced summarization that loses detail. |

---

## 3. Strategies to Overcome These Limitations

### A. Prompt Engineering Enhancements
1. **Chain‑of‑Thought (CoT) Prompts**  
   - Encourage the model to *think step by step* before giving an answer.  
   - Example: “First, list the assumptions… Then compute… Finally, give the result.”

2. **Structured Output Prompts**  
   - Ask for JSON, tables, or bullet lists explicitly.  
   - Use delimiters like ````json` to signal format.

3. **Few‑Shot Prompting**  
   - Provide a handful of examples that illustrate the desired pattern.  
   - Helps the model generalize better and reduces hallucinations.

4. **Dynamic Prompt Templates**  
   - Build prompts programmatically with placeholders (`{question}`, `{context}`) so you can inject new data without rewriting the whole prompt.

### B. System‑Level Techniques
1. **Memory Buffers / Context Windows**  
   - Store previous turns in a *conversation log* and prepend it to each new prompt.  
   - Use sliding windows or hierarchical memory (short‑term + long‑term) to stay within token limits.

2. **Retrieval Augmented Generation (RAG)**  
   - Combine the model with an external knowledge base: retrieve relevant documents, then ask the model to synthesize them.  
   - Reduces hallucinations and keeps answers grounded.

3. **Agentic Architectures**  
   - Let the LLM act as a *planner* that decides which sub‑tasks to perform (e.g., call API, fetch data).  
   - Each sub‑task can have its own prompt; the agent orchestrates them sequentially or in parallel.

4. **Post‑Processing Pipelines**  
   - Validate and correct outputs automatically: e.g., parse JSON, check for schema compliance, run sanity checks.  
   - If validation fails, re‑prompt with a *clarification* request.

5. **Fine‑Tuning / Instruction Tuning**  
   - Train the model on domain‑specific prompts or instruction sets to make it more reliable for your use case.

### C. Practical Workflow Example

| Step | What Happens | Prompt Design |
|------|--------------|---------------|
| 1. User Query | “Explain how a combustion engine works.” | `User: {query}` |
| 2. Memory Append | Add last 3 turns to context. | `Conversation: ...` |
| 3. Retrieval | Fetch top‑k Wikipedia snippets about combustion engines. | `Retrieve: {docs}` |
| 4. CoT Prompt | “First, list the main components… Then explain each component.” | Structured prompt with CoT |
| 5. Output Validation | Parse into JSON; if missing a field, ask for clarification. | Post‑process step |

---

## 4. Quick Checklist to Improve Discrete Prompts

- [ ] **Include context** (previous turns or relevant data) in the prompt.
- [ ] **Ask for structured output** (JSON, tables).
- [ ] **Add chain‑of‑thought instructions** if reasoning is needed.
- [ ] **Provide a few examples** of desired input/output pairs.
- [ ] **Use placeholders** to keep prompts reusable.
- [ ] **Validate the response** and re‑prompt if necessary.

---

## 5. Bottom Line

Discrete prompts are simple but brittle. By combining smart prompt engineering with system‑level techniques—memory management, retrieval, agentic orchestration, and validation—you can turn a single static prompt into a robust, adaptive interaction that scales to complex real‑world tasks.

**Observations**

**Question 1: Zero-Shot vs Few-Shot Prompting**

The Base Prompt answer explains the concepts clearly with examples and use cases, making the distinction between zero-shot and few-shot easy to understand; however, it is slightly repetitive and does not mention performance differences or limitations of each approach.

**Question 2: In-Context Learning Performance**

The answer effectively highlights how the number of examples and task nature affect performance, showing awareness of task complexity and example relevance.

**Question 3: Limitations of Discrete Prompts**

The Base Prompt lists key limitations and strategies to overcome them, giving practical guidance for prompt design.

## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [7]:
system_prompt = """
You are an AI assistant developed for a technology consulting firm to support Research and Development (R&D) teams in efficiently analyzing large volumes of scientific publications.

Your primary goal is to streamline the literature review process by:
- Extracting accurate and concise summaries from research papers.
- Identifying key metadata such as authors, publication year, research domain, and methodology.
- Highlighting emerging trends, insights, and innovations across related works.

When responding:
- Maintain factual accuracy and clarity at all times.
- Present insights in a structured, business-relevant, and easy-to-understand format.
- Avoid speculation or assumptions beyond the provided research content.
- If a query requires information not available in the given papers, acknowledge the limitation instead of inferring.

The objective is to help researchers and decision-makers quickly discover relevant information, reduce manual review time, and accelerate data-driven innovation.
"""


### Defining the function to Generate a Response From the LLM

In [8]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=4000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [9]:
# Define the question
question_1 = "What is the difference between zero-shot and few-shot prompting in language models?"
base_prompt_response_1 = response(system_prompt,question_1)
display(Markdown(base_prompt_response_1))

**Zero‑Shot vs. Few‑Shot Prompting**

| Aspect | Zero‑Shot Prompting | Few‑Shot Prompting |
|--------|---------------------|--------------------|
| **Definition** | The model is asked to perform a task without being given any explicit examples in the prompt. The instruction itself (e.g., “Translate this sentence…”) must be enough for the model to infer how to respond. | The prompt includes a small number of labeled examples (usually 1–5) that illustrate the desired format or behavior, followed by the new input that needs processing. |
| **Typical Use Cases** | – Simple classification or generation tasks where the instruction is clear.<br>– Scenarios with strict privacy or data‑minimization constraints (no extra text). | – Tasks requiring a specific output style or formatting.<br>– Domains where the model benefits from seeing concrete examples to reduce ambiguity. |
| **Model Behavior** | Relies entirely on its pretrained knowledge and general language understanding. May produce generic or less precise outputs if the task is subtle. | Uses the provided examples as context, enabling the model to mimic the pattern shown (e.g., “Answer in 3 bullet points”). Improves accuracy for niche formats but can be sensitive to example quality. |
| **Prompt Construction** | Concise instruction: “Summarize the following paragraph.” | Instruction + exemplar(s): <br>“Summary: This paper studies X.<br>Summary: The authors find Y.<br>Now summarize: …” |
| **Pros & Cons**

|Zero‑Shot|
|- No risk of leaking training data through examples.|
|- Faster to construct (no example selection).|
|- May be less accurate on specialized tasks.|

|Few‑Shot|
|- Higher precision when format or nuance matters.| 
|- Requires careful curation of representative examples.| 
|- Slightly longer prompts can hit token limits on large models.| 

**Bottom Line:**  
- **Zero‑shot** is “instruction only.” Use it when the task is straightforward and you want minimal prompt length.  
- **Few‑shot** adds a handful of examples to guide the model’s behavior, trading off a bit of prompt space for higher fidelity in specialized or structured tasks.

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [10]:
question_2 = "How does the performance of in-context learning change based on the number of examples provided and the nature of the task?"
base_prompt_response_2 = response(system_prompt,question_2)
display(Markdown(base_prompt_response_2))

**Impact of Example Quantity & Task Nature on In‑Context Learning (ICL) Performance**

| Factor | Typical Effect on Accuracy / Robustness | Why It Happens |
|--------|---------------------------------------|----------------|
| **Number of Examples (Few vs. Many)** | • **1–3 examples** – often enough for *classification* or *simple question answering* tasks.<br>• **4–10 examples** – noticeable performance boost; diminishing returns beyond ~10.<br>• > 10 examples – can introduce noise, longer prompts, higher memory cost; sometimes a slight drop in accuracy if irrelevant examples are included. | • The model learns task pattern from the prompt; more instances refine the implicit “template.”<br>• Excessive length may trigger token limits or prompt truncation, harming comprehension. |
| **Task Complexity** | • **Low‑complexity tasks** (e.g., sentiment classification) – modest improvement with 3–5 examples.<br>• **Medium complexity** (e.g., summarization, entity extraction) – benefits from 4–8 examples to capture structure.<br>• **High‑complexity or multi‑step reasoning** (e.g., solving math problems, code generation) – may require >10 carefully crafted examples for stable performance. | • Complex tasks have richer internal state representations; more demonstrations help the model infer the necessary sub‑procedures. |
| **Example Quality & Diversity** | • *High‑quality* (clear structure, representative of test distribution) → significant accuracy gains.<br>• *Low‑quality* or *outlier* examples can degrade performance by misleading the model. | The prompt acts as a teaching signal; noise in that signal propagates to inference. |
| **Task Domain Alignment** | • When training and target domains are closely aligned (e.g., same scientific field), fewer examples suffice.<br>• Cross‑domain transfer (medical to legal) often needs 6–12 examples to capture domain nuances. | Domain shift introduces vocabulary, style, and knowledge gaps that require extra exemplars to bridge. |
| **Prompt Length & Token Budget** | • Larger example sets consume more tokens, potentially exceeding model limits (e.g., GPT‑3’s 4k context).<br>• If truncated, the model may ignore later examples, reducing effectiveness. | The prompt must fit within token constraints; otherwise, critical cues are lost. |
| **Model Size & Architecture** | • Bigger models (175B vs. 6B) tend to generalize from fewer examples but still benefit from additional demonstrations.<br>• Some architectures (e.g., retrieval‑augmented LLMs) can compensate for limited examples by fetching related content. | Capacity and pre‑training diversity influence how much the model needs explicit guidance. |

### Practical Recommendations

| Scenario | Suggested # of Examples | Prompt Design Tips |
|----------|------------------------|--------------------|
| **Rapid prototyping / low‑complexity classification** | 3–5 | Keep examples concise, uniform format. |
| **Multi‑step reasoning or code generation** | 8–12 | Include a brief explanation for each example; maintain consistent formatting. |
| **Cross‑domain transfer** | 10–15 | Curate examples that reflect target domain terminology and style. |
| **Large‑token budget constraints** | ≤ 6 | Prioritize the most representative examples; consider summarizing longer contexts. |

### Key Takeaways

1. **More isn’t always better** – after ~10 high‑quality examples, gains plateau or even decline due to token limits and noise.
2. **Task complexity dictates quantity** – complex reasoning tasks need more exemplars than simple classification.
3. **Quality trumps quantity** – well‑structured, domain‑aligned examples provide the greatest performance lift.
4. **Monitor prompt length** – staying within the model’s context window is critical; otherwise, later examples are ignored.

*If you have a specific task or dataset in mind, let me know and I can tailor these guidelines further.*

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [11]:
question_3 = "What are the limitations of discrete prompts, and how to overcome them?"
base_prompt_response_3 = response(system_prompt,question_3)
display(Markdown(base_prompt_response_3))

## Limitations of Discrete Prompting (Single‑Prompt, “One‑Shot” Interaction)

| Limitation | Why It Happens | Typical Impact on R&D Workflow |
|------------|----------------|--------------------------------|
| **Contextual Drift** | A single prompt cannot carry all the background needed to answer a complex query. | Researchers may miss nuanced findings or misinterpret methodology due to incomplete context. |
| **Ambiguity & Over‑Generalization** | The model tries to fit a broad question into a narrow response, often glossing over important distinctions (e.g., between two similar techniques). | Critical details about experimental setups or statistical significance can be omitted. |
| **Token Limit Constraints** | Long documents cannot all be fed in one prompt; key sections may be truncated. | Loss of essential references, datasets, or supplementary material that inform downstream decisions. |
| **Error Amplification** | A single misstep (e.g., wrong citation extraction) propagates through subsequent reasoning steps. | Incorrect bibliographic metadata leads to faulty literature mapping. |
| **Static Knowledge Window** | Discrete prompts rely on the model’s static training snapshot; they cannot update in real‑time as new papers are added. | R&D teams may operate with slightly outdated knowledge, missing emerging trends. |

---

## Practical Strategies to Overcome These Limitations

| Strategy | Implementation Steps | Benefits for R&D |
|----------|----------------------|------------------|
| **Iterative Prompting (Chain‑of‑Thought)** | 1. Start with a high‑level question.<br>2. Ask the model to outline key points.<br>3. Follow up on each point with targeted prompts. | Breaks complex analysis into manageable pieces; reduces ambiguity and ensures coverage of all subtopics. |
| **Chunking & Contextual Summaries** | • Divide long PDFs into logical sections (abstract, intro, methods, results).<br>• Generate concise summaries for each chunk.<br>• Pass the concatenated summaries to the next prompt. | Keeps token usage within limits while preserving critical context; improves accuracy in metadata extraction. |
| **Hybrid Retrieval‑Augmented Generation** | • Use an information retrieval engine (e.g., ElasticSearch) to fetch relevant passages.<br>• Feed those passages into a prompt along with the query. | Provides up‑to‑date evidence and reduces hallucination of facts; aligns model output with real documents. |
| **Prompt Templates & Metadata Anchors** | Design reusable templates that explicitly request: <br>• Authors, year, DOI<br>• Research domain<br>• Methodology type (e.g., “deep learning,” “CRISPR”)<br>• Key findings & metrics | Ensures consistent extraction across papers; facilitates downstream analytics and trend mapping. |
| **Post‑Processing Validation Layer** | After the model outputs results, run rule‑based checks or a lightweight NLP pipeline to confirm: <br>– Citation format consistency<br>– Author name disambiguation<br>– Year ranges | Catches systematic errors before integration into R&D knowledge bases. |
| **Continuous Fine‑Tuning / Domain Adaptation** | Periodically fine‑tune the language model on your own corpus of peer‑reviewed papers and internal reports. | Keeps the model aligned with domain‑specific terminology and reduces hallucinations over time. |

---

## Quick Reference Flow for R&D Teams

1. **Document Ingestion** → Convert PDF to structured text (sections).
2. **Chunk & Summarize** → Generate 50–100 token summaries per section.
3. **Iterative Prompting**  
   - *Prompt 1*: “List authors, year, DOI from the summary.”  
   - *Prompt 2*: “What is the main methodology described?”  
   - *Prompt 3*: “Summarize key findings with numeric metrics.”
4. **Validate & Store** → Apply rule‑based checks; import into your literature database.
5. **Trend Analysis** → Use stored metadata to run trend queries (e.g., “top 5 emerging methods in 2023”).  

By replacing a single, monolithic prompt with this modular workflow, R&D teams can dramatically reduce manual review time while maintaining high data quality and up‑to‑date insights.

**Observation**

**Question 1 (Zero-Shot vs Few-Shot Prompting)**

The answer is well-structured with clear definitions, examples, and use cases, showing good instructional prompting.

**Question 2 (In-Context Learning Performance)**

The response demonstrates good depth and covers multiple influencing factors, but it becomes verbose, includes repeated ideas.

**Question 3 (Limitations of Discrete Prompts)**

The answer is comprehensive and lists both limitations and solutions clearly, but it is too lengthy, has overlapping points.

# Data Preparation for RAG

## Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ! unzip "/content/ResearchPaper.zip"

In [29]:
# Uploading multiple pdfs:
from glob import glob

# Path to folder with PDFs
DOC_FOLDER = "ResearchPaper/"
pdf_files = glob(DOC_FOLDER + "*.pdf")

In [30]:
pdf_files

['ResearchPaper\\AutoPrompt Eliciting Knowledge from Language Models with Automatically Generated Prompts.pdf',
 'ResearchPaper\\Language Models are Few-Shot Learners.pdf',
 'ResearchPaper\\Prompt Programming for Large Language Models.pdf',
 'ResearchPaper\\The Power of Scale for Parameter-Efficient Prompt Tuning.pdf']

In [31]:
# Curated bibliographic details (PDF metadata is often missing or wrong); keys match the PDF file names
paper_info = {
    'AutoPrompt Eliciting Knowledge from Language Models with Automatically Generated Prompts.pdf': {
        'title': 'AutoPrompt: Eliciting Knowledge from Language Models with Automatically Generated Prompts',
        'author': 'Taylor Shin, Yasaman Razeghi, Robert L. Logan IV, Eric Wallace, Sameer Singh',
        'venue': 'EMNLP 2020', 'year': '2020'},
    'Language Models are Few-Shot Learners.pdf': {
        'title': 'Language Models are Few-Shot Learners',
        'author': 'Tom B. Brown et al. (OpenAI)',
        'venue': 'NeurIPS 2020 (arXiv:2005.14165)', 'year': '2020'},
    'Prompt Programming for Large Language Models.pdf': {
        'title': 'Prompt Programming for Large Language Models: Beyond the Few-Shot Paradigm',
        'author': 'Laria Reynolds, Kyle McDonell',
        'venue': 'CHI EA 2021 (arXiv:2102.07350)', 'year': '2021'},
    'The Power of Scale for Parameter-Efficient Prompt Tuning.pdf': {
        'title': 'The Power of Scale for Parameter-Efficient Prompt Tuning',
        'author': 'Brian Lester, Rami Al-Rfou, Noah Constant',
        'venue': 'EMNLP 2021', 'year': '2021'},
}

# Load each PDF file and store the documents
all_documents = []
for pdf_file in pdf_files:
    loader = PyMuPDFLoader(pdf_file)
    documents = loader.load()
    info = paper_info.get(os.path.basename(pdf_file), {})
    for doc in documents:
        doc.metadata.update(info)                                               # Attach title, authors, venue, year to every page (and later every chunk)
    all_documents.extend(documents)


## Data Chunking

### Chunk the PDF into Manageable Text Sections

In [32]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=512,                                                             # Each chunk will have up to 512 character
)

### Split the Loaded PDF into Chunks for Further Processing

In [33]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = text_splitter.split_documents(all_documents)                  # Returns a list of chunked documents

### Check the Number of Chunks Created

In [34]:
print(f"Created {len(document_chunks)} chunks.")

Created 355 chunks.


## Generate Vector Embeddings for Text Chunks Using OpenAI

In [35]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI(base_url=OPENAI_API_BASE, api_key=OPENAI_API_KEY)    # Point the client at the local LM Studio server

In [36]:
# Initialize the OpenAI Embeddings model with API credentials
from langchain_openai import OpenAIEmbeddings                                  # Works with OpenAI-compatible servers such as LM Studio

embedding_model = OpenAIEmbeddings(
    model="text-embedding-nomic-embed-text-v1.5",                                 # Embedding model loaded in LM Studio (must match the identifier shown there)
    openai_api_key=OPENAI_API_KEY,                                                  # LM Studio ignores the key, but one is required
    openai_api_base=OPENAI_API_BASE,                                  # Local LM Studio server
    check_embedding_ctx_length=False                                            # Send raw text instead of token IDs, which LM Studio expects
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  768


Below is a sample code snippet you can refer to for using an open-source embedding model.

In [ ]:
# file_name = 'config.json'                                                       # Name of the configuration file
# with open(file_name, 'r') as file:                                              # Open the config file in read mode
#     config = json.load(file)                                                    # Load the JSON content as a dictionary
#     HF_TOKEN = config.get("HF_TOKEN")


# # Store API credentials in environment variables
# os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
# from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# # Use HuggingFaceEmbeddings wrapper with SentenceTransformer model
# embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')


## Vector Database Creation

In [37]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model                                                             # Embedding model for converting text to vectors
 )

## Retrieval and Response Generation using Vector Search

### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [38]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 5}                                                      # Retrieve top 5 most relevant documents
)

## System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [39]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support research teams in efficiently reviewing scientific literature. Your task is to provide evidence-based, concise, and relevant summaries based on the context provided from research papers.

User input will include the necessary context for you to answer their questions. This context will begin with the token:

###Context
The context contains excerpts from one or more research papers, along with associated metadata such as titles, authors, abstracts, keywords, and specific sections relevant to the query.

When crafting your response
-Use only the provided context to answer the question.
-If the answer is found in the context, respond with concise and insight-focused summaries.
-Cite every source using the metadata header that precedes each excerpt: the paper title, authors, and page number (including the venue/journal and year when they are in the header). Never invent details; write "not provided" for anything missing.
-If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


Please adhere to the following response guidelines:
-Provide clear, direct answers using only the given context.
-Do not include any additional information outside of the context.
-Avoid rephrasing or generalizing unless explicitly relevant to the question.
-If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
-If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."


Here is an example of how to structure your response:

Answer:
[Answer based on context]

Source:
[Source details with journal name, title, authors, page number or section]
"""

In [40]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from research papers, each preceded by a metadata header (title, authors, source file, page), that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [41]:
def generate_rag_response(user_input,k=5,max_tokens=3000,temperature=0.3,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    # Prefix each chunk with its metadata so the model can cite title, authors and page
    context_list = []
    for d in relevant_document_chunks:
        m = d.metadata
        header = (f"[Title: {m.get('title') or m.get('source')} | Authors: {m.get('author') or 'Unknown'} | "
                  f"Venue: {m.get('venue') or 'not provided'} | Year: {m.get('year') or 'not provided'} | "
                  f"Page: {int(m.get('page', 0)) + 1}]")
        context_list.append(header + "\n" + d.page_content)

    # Combine document chunks into a single context
    context_for_query = "\n\n".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

#### Below are sample code snippets you can refer to for downloading the open source model and implementing the function to generate a RAG-based response.

In [ ]:
# !pip install transformers torch

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# # Define model name from Hugging Face
# MISTRAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_NAME,token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL_NAME,token=HF_TOKEN)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

In [ ]:
# def generate_rag_response_mistral(user_input, k=5, max_tokens=500, temperature=0.7, top_p=0.95):
#     global qna_system_message, qna_user_message_template, retriever, model, tokenizer

#     try:

#         relevant_chunks = retriever.get_relevant_documents(query=user_input, k=k)
#         context_list = [doc.page_content for doc in relevant_chunks]
#         context = ". ".join(context_list)


#         user_message = qna_user_message_template.replace("{context}", context)
#         user_message = user_message.replace("{question}", user_input)
#         prompt = f"{qna_system_message.strip()}\n\n{user_message.strip()}"


#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_tokens,
#             temperature=temperature,
#             top_p=top_p,
#             do_sample=True
#         )

#         response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

#     except Exception as e:
#         response = f"Sorry, I encountered the following error:\n{e}"

#     return response


# Question Answering using RAG

### Question 1: What is the difference between zero-shot and few-shot prompting in language models?

In [42]:
question_1 = "What is the difference between zero-shot and few-shot prompting in language models?"
response_with_rag_1 = generate_rag_response(question_1)
display(Markdown(response_with_rag_1))

Answer:
Zero‑shot prompting gives a language model only a natural‑language instruction describing the task, with no example demonstrations. Few‑shot prompting supplies that same instruction plus a handful of task examples (typically a few dozen) before asking the model to perform the task.

Source:
- *Language Models are Few‑Shot Learners* (Tom B. Brown et al., NeurIPS 2020, p. 7) – describes zero‑shot as “the same as one‑shot except that no demonstrations are allowed” and few‑shot as “present the model with a few dozen examples in the few‑shot setting.”

### Question 2: How does the performance of in-context learning change based on the number of examples provided and the nature of the task?

In [43]:
question_2 = "How does the performance of in-context learning change based on the number of examples provided and the nature of the task?"
response_with_rag_2 = generate_rag_response(question_2)
display(Markdown(response_with_rag_2))

Answer:
In the examined work, performance improves as more examples are supplied during inference: larger language models exhibit steeper “in‑context learning curves,” meaning they benefit more from additional demonstrations or natural‑language task descriptions than smaller models (Figure 1.2). The improvement is consistent across a variety of tasks, suggesting that both the quantity of in‑context examples and the type of instruction (e.g., explicit natural‑language prompts versus few demonstrations) influence how effectively the model can adapt to a new task. However, even with many examples, in‑context learning still lags behind fine‑tuning on benchmark datasets (e.g., only 4 % on Natural Questions vs. state‑of‑the‑art scores), indicating that while scaling and richer prompts help, they do not fully close the performance gap.

Source:
- *Language Models are Few-Shot Learners* by Tom B. Brown et al., NeurIPS 2020 (arXiv:2005.14165), page 4 – Figure 1.2 description of steeper learning curves for larger models and discussion of in‑context learning performance across tasks.

### Question 3: What are the limitations of discrete prompts, and how to overcome them?

In [44]:
question_3 = "What are the limitations of discrete prompts, and how to overcome them?"
response_with_rag_3 = generate_rag_response(question_3)
display(Markdown(response_with_rag_3))

**Answer**

Discrete (hard‑coded) prompts suffer from two main limitations:

1. **Limited expressiveness and flexibility** – Because the prompt is a fixed text string, it cannot adapt to different inputs or tasks without manual redesign. This makes it difficult to encode complex task instructions or reasoning steps, and often leads to sub‑optimal performance compared with learned or dynamic prompting methods【Prompt Programming for Large Language Models: Beyond the Few‑Shot Paradigm】【ResearchPaper\Prompt Programming for Large Language Models.pdf】.

2. **High human effort and lack of automation** – Crafting an effective discrete prompt requires significant trial‑and‑error and domain expertise. The process is time‑consuming, non‑scalable, and hard to generalize across tasks【ResearchPaper\Prompt Programming for Large Language Models.pdf】【Prompt Programming for Large Language Models: Beyond the Few‑Shot Paradigm】.

**How to overcome these limitations**

- **Learned or continuous prompts** (e.g., prompt tuning, prefix tuning) replace fixed text with trainable embeddings that can be optimized for each task while keeping parameter counts low. This adds flexibility and improves task quality without manual redesign【The Power of Scale for Parameter‑Efficient Prompt Tuning】.

- **Automated prompt generation methods** such as metaprompt programming or template‑based approaches aim to generate task‑specific prompts automatically, reducing human effort and improving generalization across tasks【ResearchPaper\Prompt Programming for Large Language Models.pdf】【Prompt Programming for Large Language Models: Beyond the Few‑Shot Paradigm】.

- **Dynamic control mechanisms** (e.g., stopping criteria based on probability signals or fill‑in‑the‑blank templates) help keep autoregressive generation on track and enforce desired output formats, mitigating derailment that can occur with purely discrete prompts【ResearchPaper\Prompt Programming for Large Language Models.pdf】.

**Observation**

The answers generated from RAG are concise, well-structured, and include source references, which adds credibility and makes them trustworthy. Overall, they effectively capture key concepts in prompt engineering, providing clear explanations and relevant context, making them high-quality responses.

# Conclusion

<font size=4>Business Impact</font>

* **Enhanced Decision-Making**: Quick access to key insights and emerging trends empowers research teams to make informed, timely decisions.
* **Scalable Knowledge Management**: Structured outputs enable better indexing, retrieval, and organization of research content, improving long-term knowledge curation.

This approach lays the groundwork for building intelligent research support systems that are scalable, context-aware, and aligned with the dynamic needs of innovation-driven industries.

<font size=4>Improvement Areas</font>

* The system currently depends on the quality and consistency of input PDFs and may struggle with poorly formatted or scanned documents.
* While the LLM captures high-level insights, deeper domain-specific interpretations may require fine-tuning or human validation.

<font size=6 color='#4682B4'>Power Ahead</font>
___

# [OPTIONAL] Deployment

The deployment section is only meant to demonstrate the research assistant in action.


## Streamlit on Hugging Face

### app.py

**NOTE**: You can use this prompt to generate the code for buidling the Streamlit app. However, a few minor adjustments might be needed afterwards.

***Prompt***:

<font size=3 color="#4682B4"><b>Build a Streamlit app that lets users upload PDFs and ask questions, with answers generated via OpenAI using RAG.
</font>

In [ ]:
# %%writefile app.py

# import streamlit as st
# import os
# import tempfile
# from typing import List
# from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_core.embeddings import Embeddings
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import Chroma
# from openai import OpenAI


# # NVIDIA NIM exposes an OpenAI-compatible API
# def get_setting(name, default=None):
#     # Streamlit Cloud secrets (also exposed as env vars) first, then plain environment variables
#     try:
#         if name in st.secrets:
#             return st.secrets[name]
#     except Exception:                                                           # No secrets file configured (e.g. running locally)
#         pass
#     return os.environ.get(name, default)


# NIM_API_KEY = get_setting("API_KEY")                                            # NVIDIA NIM key (nvapi-...), kept in Streamlit secrets
# NIM_API_BASE = get_setting("API_BASE", "https://integrate.api.nvidia.com/v1")
# LLM_MODEL = "openai/gpt-oss-20b"                                                # Chat model served by NVIDIA NIM
# # Embedding models to try in order: the first one that your NIM account can actually call is used
# EMBEDDING_CANDIDATES = [
#     "nvidia/llama-3.2-nv-embedqa-1b-v1",
#     "nvidia/nv-embedqa-e5-v5",
#     "nvidia/nv-embedqa-mistral-7b-v2",
#     "nvidia/nv-embed-v1",
#     "baai/bge-m3",
#     "snowflake/arctic-embed-l",
#     "nvidia/embed-qa-4",
# ]

# st.title("LLM-Powered Research Assistant")

# # Stop with a clear message if the key is missing (OpenAI() would otherwise crash with a redacted error)
# if not NIM_API_KEY:
#     st.error("API_KEY is not set. In Streamlit Cloud open Manage app > Settings > Secrets and add: API_KEY = \"nvapi-...\"")
#     st.stop()

# # Initialize OpenAI-compatible client pointing at NVIDIA NIM
# client = OpenAI(api_key=NIM_API_KEY, base_url=NIM_API_BASE)


# class NIMEmbeddings(Embeddings):
#     """Embeddings via NVIDIA NIM. The e5 retrieval models need input_type='passage' for documents and 'query' for questions."""

#     def __init__(self, model: str, batch_size: int = 32):
#         self.model = model
#         self.batch_size = batch_size

#     def _embed(self, texts: List[str], input_type: str) -> List[List[float]]:
#         vectors = []
#         for i in range(0, len(texts), self.batch_size):
#             batch = texts[i:i + self.batch_size]
#             result = client.embeddings.create(
#                 model=self.model,
#                 input=batch,
#                 encoding_format="float",
#                 extra_body={"input_type": input_type, "truncate": "END"},
#             )
#             vectors.extend(item.embedding for item in result.data)
#         return vectors

#     def embed_documents(self, texts: List[str]) -> List[List[float]]:
#         return self._embed(texts, "passage")

#     def embed_query(self, text: str) -> List[float]:
#         return self._embed([text], "query")[0]


# # Define the system prompt for the model
# qna_system_message = """
# You are an AI assistant designed to support research teams in efficiently reviewing scientific literature. Your task is to provide evidence-based, concise, and relevant summaries based on the context provided from research papers.

# User input will include the necessary context for you to answer their questions. This context will begin with the token:

# ###Context
# The context contains excerpts from one or more research papers, along with associated metadata such as titles, authors, abstracts, keywords, and specific sections relevant to the query.

# When crafting your response
# -Use only the provided context to answer the question.
# -If the answer is found in the context, respond with concise and insight-focused summaries.
# -Cite every source using the metadata header that precedes each excerpt: the paper title, authors, and page number (including the venue/journal and year when they are in the header). Never invent details; write "not provided" for anything missing.
# -If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."


# Please adhere to the following response guidelines:
# -Provide clear, direct answers using only the given context.
# -Do not include any additional information outside of the context.
# -Avoid rephrasing or generalizing unless explicitly relevant to the question.
# -If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
# -If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."


# Here is an example of how to structure your response:

# Answer:
# [Answer based on context]

# Source:
# [Source details with journal name, title, authors, page number or section]
# """

# # Define the user message template
# qna_user_message_template = """
# ###Context
# Here are some excerpts from research papers, each preceded by a metadata header (title, authors, source file, page), that are relevant to the question mentioned below:
# {context}

# ###Question
# {question}
# """


# @st.cache_resource
# def pick_embedding_model():
#     # Listing a model on build.nvidia.com does not guarantee that this account can call it, so test each one
#     errors = []
#     for name in EMBEDDING_CANDIDATES:
#         try:
#             client.embeddings.create(model=name, input=["test"], encoding_format="float",
#                                      extra_body={"input_type": "query", "truncate": "END"})
#             return name
#         except Exception as e:
#             errors.append(f"{name}: {type(e).__name__} {str(e)[:150]}")
#     raise RuntimeError("No embedding model is available for this API key:\n" + "\n".join(errors))


# @st.cache_resource
# def load_and_process_pdfs(uploaded_files):
#     all_documents = []
#     for uploaded_file in uploaded_files:
#         with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
#             tmp_file.write(uploaded_file.getvalue())
#             tmp_file_path = tmp_file.name
#         loader = PyMuPDFLoader(tmp_file_path)
#         documents = loader.load()
#         os.remove(tmp_file_path)                                                # Clean up the temporary file
#         for doc in documents:
#             # Keep citation details on every page (and later every chunk); fall back to the file name
#             doc.metadata["source"] = uploaded_file.name
#             doc.metadata["title"] = doc.metadata.get("title") or os.path.splitext(uploaded_file.name)[0]
#             doc.metadata["author"] = doc.metadata.get("author") or "Unknown"
#         all_documents.extend(documents)

#     text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#         encoding_name="cl100k_base",
#         chunk_size=512,
#     )
#     document_chunks = text_splitter.split_documents(all_documents)

#     # Create an in-memory vector store (or use a persistent one if needed)
#     vectorstore = Chroma.from_documents(
#         document_chunks,
#         NIMEmbeddings(pick_embedding_model())
#     )
#     return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})


# def generate_rag_response(user_input, retriever, max_tokens=3000, temperature=0.3, top_p=0.95):
#     # Retrieve relevant document chunks
#     relevant_document_chunks = retriever.invoke(user_input)

#     # Prefix each chunk with its metadata so the model can cite title, authors and page
#     context_list = []
#     for d in relevant_document_chunks:
#         m = d.metadata
#         header = (f"[Title: {m.get('title') or m.get('source')} | Authors: {m.get('author') or 'Unknown'} | "
#                   f"Source file: {m.get('source')} | Page: {int(m.get('page', 0)) + 1}]")
#         context_list.append(header + "\n" + d.page_content)

#     # Combine document chunks into a single context
#     context_for_query = "\n\n".join(context_list)

#     user_message = qna_user_message_template.replace("{context}", context_for_query)
#     user_message = user_message.replace("{question}", user_input)

#     # Generate the response
#     try:
#         response = client.chat.completions.create(
#             model=LLM_MODEL,
#             messages=[
#                 {"role": "system", "content": qna_system_message},
#                 {"role": "user", "content": user_message}
#             ],
#             max_tokens=max_tokens,
#             temperature=temperature,
#             top_p=top_p
#         )
#         # gpt-oss is a reasoning model: content can be empty if reasoning used up max_tokens
#         response = (response.choices[0].message.content or "").strip()
#         if not response:
#             response = "The model returned no answer (it may have run out of tokens while reasoning). Please try again."
#     except Exception as e:
#         response = f"Sorry, I encountered the following error: \n {e}"

#     return response


# # Streamlit App
# uploaded_files = st.file_uploader("Upload PDF files", type=["pdf"], accept_multiple_files=True)

# retriever = None
# if uploaded_files:
#     st.info("Processing uploaded PDFs...")
#     try:
#         retriever = load_and_process_pdfs(uploaded_files)
#         st.success("PDFs processed and ready for questioning!")
#     except Exception as e:                                                      # Streamlit Cloud redacts uncaught errors, so show the real reason
#         st.error(f"Could not process the PDFs ({type(e).__name__}): {e}")


# if retriever:
#     user_question = st.text_input("Ask a question about the uploaded documents:")
#     if user_question:
#         with st.spinner("Generating response..."):
#             rag_response = generate_rag_response(user_question, retriever)
#             st.markdown(rag_response)

### Docker File

In [ ]:
# %%writefile Dockerfile

# FROM python:3.11-slim

# WORKDIR /app

# RUN apt-get update && apt-get install -y \
#     build-essential \
#     curl \
#     git \
#     && rm -rf /var/lib/apt/lists/*

# COPY requirements.txt ./
# RUN pip3 install --no-cache-dir -r requirements.txt

# COPY app.py ./

# # The NVIDIA NIM key is NOT baked into the image: add it as a Space secret named API_KEY
# # (optional) API_BASE defaults to https://integrate.api.nvidia.com/v1 inside app.py

# EXPOSE 8501

# HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health

# ENTRYPOINT ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0","--server.enableXsrfProtection=false"]

### requirements.txt

In [ ]:
# %%writefile requirements.txt

# langchain_community==0.3.27
# langchain==0.3.27
# chromadb==1.0.15
# pymupdf==1.26.3
# tiktoken==0.9.0
# openai==1.99.1
# streamlit>=1.35.0

## Hugging Face Setup

#### 1. Login to Hugging Face

Go to [Hugging face](https://huggingface.co) and sign up or log in to your account.

#### 2. Create a New Space

   * Navigate to [Hugging face Spaces ](https://huggingface.co/spaces).
   * Click **Create New Space**.
   * Fill in:

     * Name for your Space.
     * Space SDK: Select **Docker**.
     * Choose a Docker template: Select **Streamlit**
     * Visibility: Choose *Public* or *Private*.
   * Click **Create Space**.

#### 3. Setup OpenAI tokens




- Open your Hugging Face Space

- Click on the **“Settings”** tab at the top of the Space.

- Scroll down to the **“Repository secrets”** section.

- Click **“Add a new secret”**.

- In the **Name** field, type token name

- In the **Secret** field, paste your secret key

- Click **“Add secret”** to save it.


#### 4. Upload Your Files




   * In the new Space, click the **Files** tab.
   * Delete existing requirements.txt , DockerFile
   * Click Contribute then Upload files and add:

     * `app.py`
     * `requirements.txt`
     * `DockerFile`
   * Commit the upload.

#### 5. Build and Launch



   * Hugging Face will automatically detect the `Dockerfile` and start building the container.
   * Wait a few minutes for the build to complete and the app to go live.

#### 6. Access and Test

* Go to the App tab or the Space URL to view and test your running Streamlit app.

